# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SDlel/ml-assignments/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My baseline is a transparent priority score for pages that are visible, stale, and still have some search opportunity. It uses only observed page, search, and engagement fields from the current reporting window, not future outcomes or product flags.

The score combines visibility, freshness risk, page-one opportunity, and content depth gap. The reason codes are `stale_visible_page`, `declining_with_demand`, `thin_visible_page`, `page_one_decay_risk`, `low_ctr_visible_page`, `low_engagement_visible_page`, and `general_refresh_review`.

In [1]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "scripts").exists())

WORK_OUTPUTS = ROOT / "work" / "outputs"
WORK_OUTPUTS.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import display_path


def run_script(script_name):
    subprocess.run([sys.executable, str(ROOT / "scripts" / script_name)], cwd=ROOT, check=True)

run_script("01_prepare_features.py")
run_script("02_baseline_score.py")

feature_path = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
baseline_path = ROOT / "data" / "processed" / "baseline_refresh_queue.csv"
feature_meta = json.loads((ROOT / "data" / "processed" / "feature_metadata.json").read_text())
baseline_meta = json.loads((ROOT / "data" / "processed" / "baseline_metadata.json").read_text())
features = pd.read_csv(feature_path)
baseline_queue = pd.read_csv(baseline_path)

summary = pd.DataFrame([
    {"item": "prepared rows", "value": feature_meta["prepared_rows"]},
    {"item": "declining rows", "value": feature_meta["declining_rows"]},
    {"item": "declining rate", "value": round(feature_meta["declining_rate"], 3)},
    {"item": "baseline top score", "value": round(baseline_meta["top_score"], 3)},
    {"item": "baseline top-50 declining rate", "value": round(baseline_meta["declining_rate_top_50"], 3)},
])
summary

,item,value
0,prepared rows,30000.000
1,declining rows,16262.000
2,declining rate,0.542
3,baseline top score,0.941
4,baseline top-50 declining rate,0.340


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_scores.csv.*

The baseline queue below is sorted by `baseline_refresh_score`. I also copy the CSV into `work/outputs/` so this notebook has its own assignment artifact.

In [2]:
baseline_export = WORK_OUTPUTS / "baseline_action_scores.csv"
baseline_queue.to_csv(baseline_export, index=False)

baseline_queue.head(10)[[
    "baseline_rank", "content_id", "client_id", "baseline_refresh_score",
    "reason_codes", "suggested_action_baseline", "impressions_90d",
    "sessions_90d", "trend_direction"
]]

,baseline_rank,content_id,client_id,baseline_refresh_score,reason_codes,suggested_action_baseline,impressions_90d,sessions_90d,trend_direction
0,1,content_9532f197bbc8,client_4e07408562,0.941189,declining_with_demand|page_one_decay_risk|low_...,refresh,309192,1098,down
1,2,content_4d1fe5b32dc2,client_19581e27de,0.934889,page_one_decay_risk|low_engagement_visible_page,monitor,97999,549,stable
2,3,content_07f2e7a6f38a,client_19581e27de,0.934080,page_one_decay_risk|low_engagement_visible_page,monitor,101078,780,stable
3,4,content_e5ae436f9a16,client_4e07408562,0.933606,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,117741,522,stable
4,5,content_3430a8b94511,client_19581e27de,0.933559,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,152617,534,stable
5,6,content_cbd93118300b,client_19581e27de,0.933263,declining_with_demand|page_one_decay_risk|low_...,refresh_and_review_ctr,145292,535,down
6,7,content_9c195417f6ef,client_19581e27de,0.932991,page_one_decay_risk|low_engagement_visible_page,monitor,79146,515,stable
7,8,content_ba2acb4ebd04,client_19581e27de,0.931623,page_one_decay_risk|low_engagement_visible_page,monitor,142072,1147,stable
8,9,content_79b25654070a,client_19581e27de,0.931363,page_one_decay_risk|low_ctr_visible_page|low_e...,refresh_and_review_ctr,148737,619,stable
9,10,content_adddad39251c,client_19581e27de,0.931124,page_one_decay_risk|low_engagement_visible_page,monitor,129239,688,stable


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 are not automatic publish decisions. They are the first pages I would review because the observed metrics put them near the top of the baseline queue.

In [3]:
top20 = baseline_queue.head(20).copy()
top20["confidence_note"] = np.select(
    [
        (top20["impressions_90d"] >= 500) & (top20["sessions_90d"] >= 10),
        top20["impressions_90d"] >= 250,
    ],
    ["strong observed demand", "some observed demand"],
    default="limited observed demand",
)
top20["what_would_make_it_wrong"] = np.select(
    [
        top20["reason_codes"].str.contains("low_ctr_visible_page", regex=False),
        top20["reason_codes"].str.contains("thin_visible_page", regex=False),
        top20["reason_codes"].str.contains("stale_visible_page", regex=False),
    ],
    [
        "SERP layout or intent may explain the low CTR",
        "short content may be enough for the query intent",
        "page may still be accurate after manual review",
    ],
    default="business context may make refresh low priority",
)
review_cols = [
    "baseline_rank", "content_id", "suggested_action_baseline", "reason_codes",
    "confidence_note", "what_would_make_it_wrong", "impressions_90d", "sessions_90d",
    "trend_direction",
]
top20_review = top20[review_cols]
top20_review.to_csv(WORK_OUTPUTS / "baseline_top20_review.csv", index=False)
top20_review

,baseline_rank,content_id,suggested_action_baseline,reason_codes,confidence_note,what_would_make_it_wrong,impressions_90d,sessions_90d,trend_direction
0,1,content_9532f197bbc8,refresh,declining_with_demand|page_one_decay_risk|low_...,strong observed demand,business context may make refresh low priority,309192,1098,down
1,2,content_4d1fe5b32dc2,monitor,page_one_decay_risk|low_engagement_visible_page,strong observed demand,business context may make refresh low priority,97999,549,stable
2,3,content_07f2e7a6f38a,monitor,page_one_decay_risk|low_engagement_visible_page,strong observed demand,business context may make refresh low priority,101078,780,stable
3,4,content_e5ae436f9a16,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,strong observed demand,SERP layout or intent may explain the low CTR,117741,522,stable
4,5,content_3430a8b94511,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,strong observed demand,SERP layout or intent may explain the low CTR,152617,534,stable
5,6,content_cbd93118300b,refresh_and_review_ctr,declining_with_demand|page_one_decay_risk|low_...,strong observed demand,SERP layout or intent may explain the low CTR,145292,535,down
6,7,content_9c195417f6ef,monitor,page_one_decay_risk|low_engagement_visible_page,strong observed demand,business context may make refresh low priority,79146,515,stable
7,8,content_ba2acb4ebd04,monitor,page_one_decay_risk|low_engagement_visible_page,strong observed demand,business context may make refresh low priority,142072,1147,stable
8,9,content_79b25654070a,refresh_and_review_ctr,page_one_decay_risk|low_ctr_visible_page|low_e...,strong observed demand,SERP layout or intent may explain the low CTR,148737,619,stable
9,10,content_adddad39251c,monitor,page_one_decay_risk|low_engagement_visible_page,strong observed demand,business context may make refresh low priority,129239,688,stable


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest baseline picks are the ones where the score is high but the page has little measured engagement or the reason is mostly stale/position based. I treat these as review candidates, not proof that a refresh will help.

In [4]:
allowed_features = {
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "avg_position", "ctr",
    "content_age_days", "days_since_last_update", "trend_direction",
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
}
blocked_terms = ["future", "label", "flag", "product", "post", "after"]
used_columns = sorted(allowed_features)
suspicious_used = [col for col in used_columns if any(term in col.lower() for term in blocked_terms)]

weak_picks = baseline_queue.head(100).query("sessions_90d < 10 or impressions_90d < 250").head(10)
leakage_check = pd.DataFrame({
    "check": ["blocked terms in used columns", "uses label as feature", "uses product flags", "uses future windows"],
    "result": [len(suspicious_used), "no", "no", "no"],
})

weak_picks.to_csv(WORK_OUTPUTS / "baseline_weak_picks.csv", index=False)
leakage_check

,check,result
0,blocked terms in used columns,0
1,uses label as feature,no
2,uses product flags,no
3,uses future windows,no


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled ? markdown thinking AND code where code is requested.
- [x] Every number comes from a query, dataframe, or saved output.
- [x] I used careful language: observed / measured / directional / decision-support.
- [x] I did not include private client names, domains, URLs, titles, or keywords.